# 04 — 非线性隐变量：直接数值积分
唯一结构性修改：将 $wx+b$ 换成神经网络 $f_\theta(x)$。先验和观测噪声不变。联合密度依旧容易逐点计算，但边缘密度与后验一般无解析解。由于隐变量只有一维，仍可用网格近似积分，并直接最大化观测似然。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.stats import norm, multivariate_normal
rng = np.random.default_rng(42)
# 仅用于制造观测数据；训练代码不使用真实类别或真实参数。
centers = np.array([[-2., 0.], [2., 0.], [0., 3.]])
y = centers[rng.integers(0, 3, 900)] + 0.45*rng.normal(size=(900, 2))
def compare(samples):
    fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
    for a, data, title in zip(ax, [y, samples], ['Observed', 'Generated']):
        a.scatter(*data.T, s=5, alpha=.5); a.set_title(title); a.set_aspect('equal')
    plt.show()


In [ ]:
import torch
from torch import nn
torch.set_num_threads(1); torch.manual_seed(42)
Y = torch.tensor(y,dtype=torch.float32)
sigma = .45
decoder = nn.Sequential(nn.Linear(1,32),nn.Tanh(),nn.Linear(32,32),nn.Tanh(),nn.Linear(32,2))
grid = torch.linspace(-6,6,241)[:,None]; dx = float(grid[1]-grid[0])
log_prior = torch.distributions.Normal(0.,1.).log_prob(grid[:,0])
def log_joint(obs, grid, log_prior):
    return torch.distributions.Normal(decoder(grid),sigma).log_prob(obs[:,None,:]).sum(-1)+log_prior
opt = torch.optim.Adam(decoder.parameters(),lr=.01)
history=[]
for _ in range(350):
    batch=Y[torch.randint(len(Y),(128,))]
    lj=log_joint(batch,grid,log_prior)
    ll=(torch.logsumexp(lj,1)+np.log(dx)).mean()
    opt.zero_grad(); (-ll).backward(); opt.step(); history.append(ll.item())
plt.plot(history); plt.ylabel('mean log likelihood (grid)'); plt.show()
with torch.no_grad():
    lj=log_joint(Y[:1],grid,log_prior)[0]
    log_z=torch.logsumexp(lj,0)+np.log(dx)
    plt.plot(grid[:,0],torch.exp(lj-log_z)); plt.title('Posterior density for first observation'); plt.show()
    compare((decoder(torch.randn(len(y),1))+sigma*torch.randn(len(y),2)).numpy())

In [ ]:
with torch.no_grad():
    for size in [61,121,241,481,961]:
        g=torch.linspace(-6,6,size)[:,None]
        lp=torch.distributions.Normal(0.,1.).log_prob(g[:,0])
        val=torch.logsumexp(log_joint(Y[:10],g,lp),1)+torch.log(g[1,0]-g[0,0])
        print(size, val.mean().item())
    torch.save(decoder.state_dict(),'decoder_grid.pt')

每维 241 个网格点，二维就需要 241² 个点，十维需要 241¹⁰ 个点。这里是近似积分，不是精确后验。下一章不改变生成模型，而是用另一个网络近似后验。